# ch04 Bonus 05：混合专家（Mixture of Experts, MoE）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/07_moe
> **参考真实模型**：Mixtral / DeepSeek-MoE / Qwen-MoE / GPT-4（传闻）

## 一句话

用**路由器**为每个 token 选 top-k 个专家子网络，其余专家跳过。模型总参数量大，但每次推理只激活一小部分，实现'参数多、算得快'。

## 为什么需要 MoE

稠密模型（dense）的每个 token 都过全部参数，参数量和计算量绑定。MoE 解耦了二者：

| 指标 | 稠密模型 | MoE |
|------|---------|-----|
| 总参数量 | N | **N×E**（E 个专家） |
| 单 token 激活 | N | **N×top_k/E**（只算被选中的 k 个） |

例如 Mixtral 8×7B：总参数 47B，但每个 token 只激活 2 个专家 ≈ 13B 的算力。

## 核心组件

1. **Router（门控）**：一个线性层输出每个 token 对各专家的偏好分数
2. **Top-k 选择**：只保留分数最高的 k 个专家，对这 k 个分数做 softmax 加权
3. **专家**：每个专家是一个独立的 FFN（这里简化为 Linear）

> 训练时还需**负载均衡损失**（load balancing loss），防止路由器总选同几个专家。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MoELayer(nn.Module):
    """混合专家层：每个 token 被路由到 top_k 个专家，加权求和。"""

    def __init__(self, d_in, d_out, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        # 路由器：d_in → num_experts 的打分
        self.router = nn.Linear(d_in, num_experts, bias=False)
        # 专家：每个是一个独立子网络
        self.experts = nn.ModuleList([
            nn.Linear(d_in, d_out) for _ in range(num_experts)
        ])

    def forward(self, x):
        b, n, d = x.shape
        x_flat = x.view(b * n, d)                       # [b*n, d]
        router_logits = self.router(x_flat)             # [b*n, num_experts]
        # 选 top_k 个专家
        topk_logits, topk_idx = router_logits.topk(self.top_k, dim=-1)
        # 仅对选中的 k 个分数做 softmax（稀疏加权）
        topk_weights = F.softmax(topk_logits, dim=-1)   # [b*n, top_k]

        # 计算输出：对每个被选中的专家，加权其结果
        out = torch.zeros_like(x_flat)
        for i in range(self.top_k):
            expert_idx_per_token = topk_idx[:, i]        # 每 token 选的第 i 个专家
            weight = topk_weights[:, i:i+1]
            for e in range(self.num_experts):
                mask = (expert_idx_per_token == e)
                if mask.any():
                    out[mask] += weight[mask] * self.experts[e](x_flat[mask])
        return out.view(b, n, -1), router_logits        # router_logits 供算负载均衡损失

## 2. 运行 MoE：观察路由与负载

In [ ]:
torch.manual_seed(123)
batch, seq, dim = 2, 16, 768
x = torch.randn(batch, seq, dim)

moe = MoELayer(dim, dim, num_experts=4, top_k=2)
out, router_logits = moe(x)
print(f"MoE 输出: {tuple(out.shape)}")

# 统计专家负载（每个专家被选中几次）
_, topk_idx = router_logits.topk(2, dim=-1)
counts = torch.bincount(topk_idx.flatten(), minlength=4)
total = counts.sum().item()
print(f"\n4 个专家的负载（每 token 选 2 个，共 {total} 次分配）：")
for e in range(4):
    bar = '█' * int(counts[e] / total * 40)
    print(f"  专家 {e}: {counts[e].item():2d} ({100*counts[e]/total:4.1f}%) {bar}")
print(f"\n💡 理想负载应均匀（各 50%）；训练时用负载均衡损失纠正偏差。")

## 3. 参数量 vs 激活量

In [ ]:
total_params = sum(p.numel() for p in moe.parameters())
# 每个专家独立，激活量 = top_k/num_experts
active_ratio = moe.top_k / moe.num_experts
print(f"总参数量: {total_params:,}（含 {moe.num_experts} 个专家）")
print(f"单 token 激活比例: top_k/num_experts = {moe.top_k}/{moe.num_experts} = {active_ratio:.0%}")
print(f"\n这就是 MoE 的核心价值：参数多，但每次只算一小部分。")

---
> 📌 本 notebook 实现 MoE 路由与稀疏激活，并观察专家负载。
> 含负载均衡损失的完整训练实现见官方 `ch04/07_moe`。